# Age detection from a face crop

Using OpenCV's pretrained age estimation Caffe model (`age_net.caffemodel` + `age_deploy.prototxt`). The model is a Levi-Hassner CNN that buckets age into 8 ranges, not a regression head, so the output is one of `(0-2)`, `(4-6)`, `(8-12)`, `(15-20)`, `(25-32)`, `(38-43)`, `(48-53)`, `(60-100)`.

You have to download the two model files (totaling ~45MB) and point `MODEL_DIR` at them. I'm not committing the weights to the repo. If the files aren't there, the notebook prints what's missing and skips the inference cell.

In [1]:
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("../../data/sample_images")
MODEL_DIR = Path("../../data/age_model")  # drop age_net.caffemodel + age_deploy.prototxt here
ART = Path("artifacts"); ART.mkdir(exist_ok=True)

AGE_BUCKETS = ['(0-2)', '(4-6)', '(8-12)', '(15-20)',
               '(25-32)', '(38-43)', '(48-53)', '(60-100)']
MODEL_MEAN = (78.4263377603, 87.7689143744, 114.895847746)  # from the Levi-Hassner paper

proto = MODEL_DIR / "age_deploy.prototxt"
weights = MODEL_DIR / "age_net.caffemodel"
have_model = proto.exists() and weights.exists()
print("have age model:", have_model)
if not have_model:
    print(f"missing: drop age_deploy.prototxt and age_net.caffemodel into {MODEL_DIR}")

have age model: False
missing: drop age_deploy.prototxt and age_net.caffemodel into ../../data/age_model


In [2]:
# Haar cascade for finding a face to crop. Same default cascade as project 68.
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def crop_face(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    boxes = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60,60))
    if len(boxes) == 0:
        return None, None
    # pick the biggest box; argmax on area
    areas = [w*h for (x,y,w,h) in boxes]
    x,y,w,h = boxes[int(np.argmax(areas))]
    return img_bgr[y:y+h, x:x+w], (x,y,w,h)

In [3]:
def predict_age(face_bgr):
    blob = cv2.dnn.blobFromImage(face_bgr, 1.0, (227, 227), MODEL_MEAN, swapRB=False)
    net = cv2.dnn.readNetFromCaffe(str(proto), str(weights))
    net.setInput(blob)
    probs = net.forward()[0]
    idx = int(np.argmax(probs))
    return AGE_BUCKETS[idx], float(probs[idx])

## Demo

If model weights are present and `data/sample_images/` has at least one face, run it. Otherwise the notebook prints what's missing — I'd rather have a deliberate "skipped" message than a fake number.

In [4]:
paths = sorted([p for p in DATA.glob("*") if p.suffix.lower() in {".jpg",".jpeg",".png"}])

if not have_model:
    print("Skipping inference: no model weights. See top cell.")
elif not paths:
    print("Skipping inference: no images in data/sample_images/.")
else:
    fig, axes = plt.subplots(1, min(3, len(paths)), figsize=(4*min(3,len(paths)), 4))
    if min(3, len(paths)) == 1: axes = [axes]
    for ax, p in zip(axes, paths[:3]):
        img = cv2.imread(str(p))
        face, box = crop_face(img)
        if face is None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(f"{p.name}\nno face found"); ax.axis("off")
            continue
        bucket, conf = predict_age(face)
        x,y,w,h = box
        drawn = img.copy()
        cv2.rectangle(drawn, (x,y), (x+w,y+h), (0,220,0), 2)
        cv2.putText(drawn, f"{bucket} ({conf:.2f})", (x, y-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,220,0), 2)
        ax.imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
        ax.set_title(p.name); ax.axis("off")
    plt.tight_layout(); plt.savefig(ART/"ages.png", dpi=110); plt.show()

Skipping inference: no model weights. See top cell.


## Notes

A few things I noticed playing with this:

- The Levi-Hassner model was trained on Adience, which skews quite a bit toward American/European faces. Don't expect great calibration on faces it didn't see much of in training.
- Bucket boundaries are wide. The model can be "right" on the bucket but 5 years off on the actual age, and you'd never know from this output.
- It's also bad on faces wearing sunglasses or with heavy makeup — both heavily perturb the periocular region, which is one of the strongest age cues.

If I wanted a real number rather than a bucket, the obvious upgrade is something like the `nateraw/vit-age-classifier` HuggingFace model or a small DEX-style regression head. Out of scope here.